In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path.cwd().parent / 'data' / 'interim'
DATA_START = pd.Timestamp('2030-12-01', tz='UTC')
DATA_END   = pd.Timestamp('2032-01-14', tz='UTC')


In [ ]:
# Load all six tables
clients        = pd.read_csv(DATA / 'clients.csv')
transactions   = pd.read_csv(DATA / 'transactions.csv')
messages       = pd.read_csv(DATA / 'messages.csv')
templates      = pd.read_csv(DATA / 'message_templates.csv')
conversions    = pd.read_csv(DATA / 'conversions.csv')
business_units = pd.read_csv(DATA / 'business_units.csv')

clients['registration_date'] = pd.to_datetime(clients['registration_date'], utc=True)
transactions['date']         = pd.to_datetime(transactions['date'], utc=True)
messages['send_date']        = pd.to_datetime(messages['send_date'], utc=True)
conversions['date']          = pd.to_datetime(conversions['date'], utc=True)

for name, df in [('clients', clients), ('transactions', transactions),
                  ('messages', messages), ('templates', templates),
                  ('conversions', conversions), ('business_units', business_units)]:
    print(f'{name:>18}: {len(df):,} rows')


In [ ]:
# Unique locations and categories visited per client (all-time)
tx_with_bu = transactions.merge(
    business_units[['id', 'location', 'category']],
    left_on='business_unit_id', right_on='id', how='left'
)
cross = tx_with_bu.groupby('client_id').agg(
    unique_locations=('location', 'nunique'),
    unique_categories=('category', 'nunique'),
).reset_index()

mono  = (cross['unique_locations'] == 1).sum()
multi = (cross['unique_locations'] >  1).sum()
print(f'Single-format clients:  {mono:,} ({mono/len(cross)*100:.1f}%)')
print(f'Multi-format clients:   {multi:,} ({multi/len(cross)*100:.1f}%)')
print(cross['unique_locations'].value_counts().sort_index().to_string())


In [ ]:
# New clients registered within the dataset window
new_clients = clients[
    clients['registration_date'].between(DATA_START, DATA_END)
].copy()
new_clients['observation_days'] = (DATA_END - new_clients['registration_date']).dt.days

print(f'New clients:            {len(new_clients):,}')
print(f'Min observation (days): {new_clients["observation_days"].min()}')
print(f'Max observation (days): {new_clients["observation_days"].max()}')
new_clients.groupby(new_clients['registration_date'].dt.to_period('M'))['observation_days'].median().astype(int).rename('median_obs').to_frame()


In [ ]:
# Left-censoring: clients who register AT the point-of-sale during a purchase
tx_new = transactions[transactions['client_id'].isin(new_clients['client_id'])].copy()
first_tx = tx_new.groupby('client_id')['date'].min().reset_index()
first_tx.columns = ['client_id', 'first_tx_date']

lc = new_clients[['client_id', 'registration_date']].merge(first_tx, on='client_id', how='inner')
lc['diff_min'] = (lc['registration_date'] - lc['first_tx_date']).dt.total_seconds() / 60

early = lc[lc['diff_min'] > 0]  # tx before registration
print(f'Clients with tx before registration: {len(early):,} ({len(early)/len(lc)*100:.1f}%)')
bins   = [0, 1, 5, 60, 1440, float('inf')]
labels = ['<1 min', '1-5 min', '5-60 min', '1-24 h', '>24 h']
print(pd.cut(early['diff_min'], bins=bins, labels=labels).value_counts().sort_index().to_string())


In [ ]:
# Cumulative share of new clients with a second transaction by day X
tx_new_sorted = tx_new.merge(new_clients[['client_id', 'registration_date']], on='client_id', how='left')
tx_new_sorted['days_since_reg'] = (tx_new_sorted['date'] - tx_new_sorted['registration_date']).dt.days
tx_new_sorted = tx_new_sorted.sort_values(['client_id', 'days_since_reg'])
tx_new_sorted['rank'] = tx_new_sorted.groupby('client_id').cumcount() + 1
second_tx = tx_new_sorted[tx_new_sorted['rank'] == 2][['client_id', 'days_since_reg']]
n_clients = len(new_clients)

print('Cumulative share with 2nd tx by day X:')
for d in [30, 45, 60, 90, 120, 180]:
    pct = (second_tx['days_since_reg'] <= d).sum() / n_clients * 100
    print(f'  day {d:>3}: {pct:.1f}%')


In [ ]:
# Message template segment distribution
print(templates['segment'].value_counts().to_string())
